# Dataflow Gen1 to Gen2 Upgrade Accelerator (Current Fabric APIs - Enhanced UI)

This notebook uses the current Fabric Gen1-to-Gen2 upgrade APIs with enhanced UI features:
- Search capabilities for workspace and dataflow selection
- Color-coded readiness results based on named statuses
- Readiness: `GET /v1/workspaces/{workspaceId}/dataflows/gen1UpgradeReadinessResults`
- Migration: `POST /v1/workspaces/{workspaceId}/dataflows/gen1Upgrade` and standard Fabric LRO polling
- Upgrade batches contain 1-50 dataflows from a single workspace

In [ ]:
import time
import requests
import pandas as pd
import ipywidgets as widgets
from notebookutils import mssparkutils
from IPython.display import display, HTML

REQUEST_TIMEOUT_SECONDS = 30
READINESS_PAGE_SIZE = 30
RATE_LIMIT_DELAY_SECONDS = 0.5
POLL_INTERVAL_SECONDS = 5
POLL_TIMEOUT_SECONDS = 1800

In [ ]:
FABRIC_API_BASE_URL = "https://api.fabric.microsoft.com/v1"


def GetToken() -> str:
    return mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")


def Request(method: str, url: str, headers: dict, params: dict = None, json_body: dict = None):
    response = requests.request(
        method=method,
        url=url,
        headers=headers,
        params=params,
        json=json_body,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    return response


def GetWorkspaces(headers: dict, useAdminAPI: str = "Y", page_size: int = 1000) -> pd.DataFrame:
    use_admin = str(useAdminAPI).strip().upper() == "Y"
    all_rows = []

    if use_admin:
        base_url = "https://api.powerbi.com/v1.0/myorg/admin/groups"
        skip = 0
        while True:
            params = {
                "$top": str(page_size),
                "$skip": str(skip),
                "$filter": "type ne 'Personal' and type ne 'PersonalGroup'",
            }
            resp = Request("GET", base_url, headers=headers, params=params)
            items = resp.json().get("value", [])
            all_rows.extend(items)
            if len(items) < page_size:
                break
            skip += page_size
    else:
        next_url = "https://api.powerbi.com/v1.0/myorg/groups?$top=5000"
        while next_url:
            resp = Request("GET", next_url, headers=headers)
            payload = resp.json()
            all_rows.extend(payload.get("value", []))
            next_url = payload.get("@odata.nextLink")

    df = pd.DataFrame(all_rows)
    if df.empty:
        return df

    if "type" in df.columns:
        df = df[~df["type"].isin(["Personal", "PersonalGroup"])].copy()

    keep_cols = ["id", "name", "isOnDedicatedCapacity", "capacityId"]
    keep_cols = [column for column in keep_cols if column in df.columns]
    return df[keep_cols].drop_duplicates(subset=["id"])


def GetWorkspaceReadiness(fabric_api_base_url: str, workspace_id: str, headers: dict, page_size: int = READINESS_PAGE_SIZE) -> list:
    all_items = []
    continuation_token = None

    while True:
        url = f"{fabric_api_base_url}/workspaces/{workspace_id}/dataflows/gen1UpgradeReadinessResults"
        params = {"pageSize": page_size}
        if continuation_token:
            params["continuationToken"] = continuation_token

        resp = Request("GET", url, headers=headers, params=params)
        payload = resp.json()
        items = payload.get("value", []) if isinstance(payload, dict) else []
        all_items.extend(items)

        continuation_token = payload.get("continuationToken") if isinstance(payload, dict) else None
        if not continuation_token:
            break

    return all_items


def EvaluateWorkspaces(fabric_api_base_url: str, workspace_ids: list, headers: dict, workspace_name_lookup: dict = None) -> pd.DataFrame:
    rows = []
    workspace_name_lookup = workspace_name_lookup or {}

    for idx, workspace_id in enumerate(workspace_ids):
        workspace_name = workspace_name_lookup.get(workspace_id)
        try:
            items = GetWorkspaceReadiness(fabric_api_base_url, workspace_id, headers)
        except Exception as error:
            rows.append({
                "workspaceObjectId": workspace_id,
                "workspaceName": workspace_name,
                "dataflowObjectId": None,
                "dataflowName": None,
                "status": "Unknown",
                "reasonStatus": None,
                "reasonCode": "READINESS_ERROR",
                "reasonMessage": str(error),
            })
            continue

        if idx < len(workspace_ids) - 1:
            time.sleep(RATE_LIMIT_DELAY_SECONDS)

        if not items:
            rows.append({
                "workspaceObjectId": workspace_id,
                "workspaceName": workspace_name,
                "dataflowObjectId": None,
                "dataflowName": None,
                "status": None,
                "reasonStatus": None,
                "reasonCode": None,
                "reasonMessage": None,
            })
            continue

        for item in items:
            reasons = item.get("reasons") if isinstance(item.get("reasons"), list) else []
            base = {
                "workspaceObjectId": workspace_id,
                "workspaceName": workspace_name,
                "dataflowObjectId": item.get("id"),
                "dataflowName": item.get("displayName"),
                "status": item.get("status"),
            }

            if not reasons:
                row = dict(base)
                row.update({"reasonStatus": None, "reasonCode": None, "reasonMessage": None})
                rows.append(row)
            else:
                for reason in reasons:
                    row = dict(base)
                    row.update({
                        "reasonStatus": reason.get("status"),
                        "reasonCode": reason.get("code"),
                        "reasonMessage": reason.get("message"),
                    })
                    rows.append(row)

    return pd.DataFrame(rows)


def SubmitMigrationJob(fabric_api_base_url: str, workspace_id: str, headers: dict, dataflow_ids: list) -> str:
    unique_dataflow_ids = list(dict.fromkeys(dataflow_ids))
    if not 1 <= len(unique_dataflow_ids) <= 50:
        raise ValueError("Each upgrade request must contain between 1 and 50 dataflows.")

    url = f"{fabric_api_base_url}/workspaces/{workspace_id}/dataflows/gen1Upgrade"
    body = {"dataflows": [{"id": dataflow_id} for dataflow_id in unique_dataflow_ids]}
    resp = Request("POST", url, headers=headers, json_body=body)
    operation_id = resp.headers.get("x-ms-operation-id") or resp.json().get("operationId")
    if not operation_id:
        raise RuntimeError(f"Operation ID missing from response: {resp.text}")
    return operation_id


def GetOperationStatus(fabric_api_base_url: str, headers: dict, operation_id: str) -> dict:
    url = f"{fabric_api_base_url}/operations/{operation_id}"
    return Request("GET", url, headers=headers).json()


def GetOperationResult(fabric_api_base_url: str, headers: dict, operation_id: str) -> dict:
    url = f"{fabric_api_base_url}/operations/{operation_id}/result"
    return Request("GET", url, headers=headers).json()


def PollOperation(fabric_api_base_url: str, headers: dict, operation_id: str) -> dict:
    terminal = {"Succeeded", "Failed"}
    start = time.time()

    while True:
        status_payload = GetOperationStatus(fabric_api_base_url, headers, operation_id)
        status = status_payload.get("status", "Undefined")
        percent = status_payload.get("percentComplete")
        if percent is None:
            print(f"Operation {operation_id}: {status}")
        else:
            print(f"Operation {operation_id}: {status} ({percent}%)")

        if status in terminal:
            return status_payload

        if time.time() - start > POLL_TIMEOUT_SECONDS:
            raise TimeoutError(f"Operation polling timed out after {POLL_TIMEOUT_SECONDS} seconds")

        time.sleep(POLL_INTERVAL_SECONDS)

In [ ]:
def GetPowerBIToken() -> str:
    return mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")

In [ ]:
# Helper function for color-coding evaluation results based on the Fabric readiness status
def get_status_color(status_value):
    status_colors = {
        "ReadyToMigrate": "background-color: #90EE90",
        "NeedsAttention": "background-color: #FFFFE0",
        "UpgradeUnavailable": "background-color: #FFB6C6",
        "Unknown": "background-color: #E0E0E0",
    }
    return status_colors.get(status_value, "background-color: white")


def color_code_dataframe_by_status(df, status_col="status"):
    """Apply color coding to entire rows based on a named status column."""
    if status_col not in df.columns:
        return df.style

    def row_colorer(row):
        color = get_status_color(row.get(status_col))
        return [color] * len(row)

    return df.style.apply(row_colorer, axis=1)

In [ ]:
print("Authenticating for Fabric APIs...")
token = GetToken()
fabric_api_base_url = FABRIC_API_BASE_URL
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}
power_bi_headers = {
    "Authorization": f"Bearer {GetPowerBIToken()}",
    "Content-Type": "application/json",
}
print(f"Fabric API base URL: {fabric_api_base_url}")

vWorkspaceSummary = GetWorkspaces(headers=power_bi_headers, useAdminAPI="Y") # if you are not a Fabric Admin, change to useAdminAPI = "N"
workspace_name_lookup = dict(zip(vWorkspaceSummary["id"], vWorkspaceSummary["name"]))
print(f"Workspaces retrieved: {len(vWorkspaceSummary)}")
vWorkspaceSummary.head()

In [ ]:
# Enhanced Workspace Selection with Search
if vWorkspaceSummary.empty:
    print("No workspaces available to select.")
else:
    workspace_options = [
        (f"{row['name']} | {row['id']}", row['id'])
        for _, row in vWorkspaceSummary.sort_values("name").iterrows()
    ]

    # Create search box
    ws_search = widgets.Text(
        placeholder='Search workspaces by name or ID...',
        description='Search:',
        layout=widgets.Layout(width='95%')
    )

    # Create main selection widget
    workspace_picker = widgets.SelectMultiple(
        options=workspace_options,
        value=(),
        rows=min(20, len(workspace_options)),
        description="Workspaces:",
        layout=widgets.Layout(width="95%", height="350px"),
    )

    btn_select_all = widgets.Button(description="Select All", button_style="info")
    btn_clear = widgets.Button(description="Clear", button_style="warning")
    btn_confirm = widgets.Button(description="Confirm Selection", button_style="success")
    ws_output = widgets.Output()
    ws_count = widgets.HTML(value="")

    def _on_search_change(change):
        """Filter workspace options based on search text"""
        search_text = change['new'].lower()
        if search_text:
            filtered_options = [
                (label, value) for label, value in workspace_options
                if search_text in label.lower()
            ]
        else:
            filtered_options = workspace_options
        
        workspace_picker.options = filtered_options
        ws_count.value = f"<i>Showing {len(filtered_options)} of {len(workspace_options)} workspaces</i>"

    def _on_select_all(_):
        workspace_picker.value = tuple([item[1] for item in workspace_picker.options])

    def _on_clear(_):
        workspace_picker.value = tuple()

    def _on_confirm(_):
        global SelectedWorkspaces
        selected_ids = list(workspace_picker.value)
        SelectedWorkspaces = vWorkspaceSummary[vWorkspaceSummary["id"].isin(selected_ids)].copy()
        with ws_output:
            ws_output.clear_output(wait=True)
            print(f"✓ Selected workspaces: {len(SelectedWorkspaces)}")
            if not SelectedWorkspaces.empty:
                display(SelectedWorkspaces[["id", "name"]])

    ws_search.observe(_on_search_change, names='value')
    btn_select_all.on_click(_on_select_all)
    btn_clear.on_click(_on_clear)
    btn_confirm.on_click(_on_confirm)

    display(widgets.HTML("<h4>Select Workspaces</h4>"))
    display(ws_search)
    display(ws_count)
    display(workspace_picker)
    display(widgets.HBox([btn_select_all, btn_clear, btn_confirm]))
    display(ws_output)

In [ ]:
if "SelectedWorkspaces" in globals() and isinstance(SelectedWorkspaces, pd.DataFrame) and not SelectedWorkspaces.empty:
    workspace_ids = SelectedWorkspaces["id"].tolist()
    print(f"Using selected workspaces: {len(workspace_ids)}")
else:
    workspace_ids = vWorkspaceSummary["id"].tolist()
    print(f"No selection provided. Using all workspaces: {len(workspace_ids)}")

workspace_name_lookup = dict(zip(vWorkspaceSummary["id"], vWorkspaceSummary["name"]))
vEvaluationSummary = EvaluateWorkspaces(
    fabric_api_base_url,
    workspace_ids,
    headers,
    workspace_name_lookup=workspace_name_lookup,
)
print(f"Total evaluation rows: {len(vEvaluationSummary)}")
display(color_code_dataframe_by_status(vEvaluationSummary.head(20)))

In [ ]:
# Enhanced Dataflow Selection with Search
eval_with_dataflows = vEvaluationSummary[vEvaluationSummary["dataflowObjectId"].notna()].copy()
display_columns = [
    "workspaceObjectId",
    "workspaceName",
    "dataflowObjectId",
    "dataflowName",
    "status",
    "reasonCode",
    "reasonMessage",
]
existing_columns = [column for column in display_columns if column in eval_with_dataflows.columns]
eval_with_dataflows = eval_with_dataflows[existing_columns]

workspace_list = sorted(eval_with_dataflows["workspaceName"].dropna().unique())
output_area = widgets.Output()


def _fmt(value):
    return "" if pd.isna(value) else str(value)


def _get_status_badge(status_value):
    status_styles = {
        "ReadyToMigrate": ("#90EE90", "Ready to migrate"),
        "NeedsAttention": ("#FFFFE0", "Needs attention"),
        "UpgradeUnavailable": ("#FFB6C6", "Upgrade unavailable"),
        "Unknown": ("#E0E0E0", "Unknown"),
    }
    color, label = status_styles.get(_fmt(status_value), ("#E0E0E0", _fmt(status_value) or "-"))
    return f'<span style="background-color: {color}; padding: 2px 6px; border-radius: 3px; font-weight: bold;">{label}</span>'


def _build_dataflow_label(dataflow_row: pd.Series) -> str:
    dataflow_name = _fmt(dataflow_row.get("dataflowName")) or "[Unnamed Dataflow]"
    status = _get_status_badge(dataflow_row.get("status"))
    reason_code = _fmt(dataflow_row.get("reasonCode")) or "-"
    return f"<span style='font-weight:700; color:#0078D4;'>{dataflow_name}</span> <span>| Status: {status} | Reason: {reason_code}</span>"


def on_confirm_clicked(_):
    selected_dataflows_list = []
    with output_area:
        output_area.clear_output(wait=True)
        for workspace_name in workspace_list:
            for dataflow_checkbox in workspace_checkboxes.get(workspace_name, []):
                if not dataflow_checkbox.value:
                    continue
                dataflow_rows = eval_with_dataflows[
                    (eval_with_dataflows["workspaceName"] == workspace_name)
                    & (eval_with_dataflows["dataflowObjectId"] == dataflow_checkbox.dataflow_id)
                ]
                if not dataflow_rows.empty:
                    selected_dataflows_list.append(dataflow_rows.iloc[0])

    global SelectedDataflows
    SelectedDataflows = pd.DataFrame(selected_dataflows_list)
    with output_area:
        print(f"Selected {len(SelectedDataflows)} dataflow(s) for migration.")
        if not SelectedDataflows.empty:
            display(color_code_dataframe_by_status(SelectedDataflows))


workspace_checkboxes = {}
workspace_boxes = []

for workspace_name in workspace_list:
    workspace_label = widgets.Label(f"Workspace: {workspace_name}", layout=widgets.Layout(width="98%"))
    workspace_dataflows = eval_with_dataflows[
        eval_with_dataflows["workspaceName"] == workspace_name
    ].drop_duplicates(subset=["dataflowObjectId"], keep="first")

    dataflow_search = widgets.Text(
        placeholder="Filter dataflows...",
        description="",
        layout=widgets.Layout(width="95%", margin="5px 0px"),
    )
    dataflow_checkboxes = []
    dataflow_rows_map = {}

    for _, dataflow_row in workspace_dataflows.iterrows():
        dataflow_id = dataflow_row.get("dataflowObjectId")
        if pd.isna(dataflow_id):
            continue
        checkbox = widgets.Checkbox(value=False, description="", indent=False, layout=widgets.Layout(width="30px"))
        checkbox.dataflow_id = dataflow_id
        dataflow_checkboxes.append(checkbox)
        label_widget = widgets.HTML(value=_build_dataflow_label(dataflow_row), layout=widgets.Layout(width="calc(100% - 40px)"))
        row_widget = widgets.HBox([checkbox, label_widget], layout=widgets.Layout(width="98%"))
        dataflow_rows_map[dataflow_id] = {
            "row_widget": row_widget,
            "name": _fmt(dataflow_row.get("dataflowName")).lower(),
        }

    workspace_checkboxes[workspace_name] = dataflow_checkboxes
    dataflow_container = widgets.VBox(
        [row["row_widget"] for row in dataflow_rows_map.values()],
        layout=widgets.Layout(width="98%"),
    )

    def _make_search_handler(container, rows_map):
        def _on_dataflow_search_change(change):
            search_text = change["new"].lower()
            visible_rows = [
                row_info["row_widget"]
                for row_info in rows_map.values()
                if search_text in row_info["name"]
            ]
            container.children = visible_rows or [widgets.HTML("<i>No dataflows match search criteria</i>")]
        return _on_dataflow_search_change

    dataflow_search.observe(_make_search_handler(dataflow_container, dataflow_rows_map), names="value")
    workspace_boxes.append(
        widgets.VBox(
            [workspace_label, dataflow_search, dataflow_container],
            layout=widgets.Layout(width="99%", border="1px solid #ddd", padding="10px", margin="5px"),
        )
    )

confirm_button = widgets.Button(description="Confirm Selections", button_style="success")
confirm_button.on_click(on_confirm_clicked)

ui = widgets.VBox([
    widgets.HTML("<h4>Select Dataflows to Migrate</h4>"),
    widgets.VBox(workspace_boxes, layout=widgets.Layout(width="100%", max_height="600px", overflow_y="auto", overflow_x="auto")),
    confirm_button,
    output_area,
], layout=widgets.Layout(width="100%"))

display(ui)

In [ ]:
if "SelectedDataflows" not in globals() or SelectedDataflows.empty:
    print("No dataflows selected. Use the selection cell above first.")
else:
    selected_migrations = SelectedDataflows.dropna(
        subset=["workspaceObjectId", "dataflowObjectId"]
    ).drop_duplicates(subset=["workspaceObjectId", "dataflowObjectId"])
    all_result_rows = []

    for workspace_id, workspace_dataflows in selected_migrations.groupby("workspaceObjectId"):
        dataflow_ids = workspace_dataflows["dataflowObjectId"].tolist()
        for start_index in range(0, len(dataflow_ids), 50):
            batch_ids = dataflow_ids[start_index:start_index + 50]
            operation_id = SubmitMigrationJob(
                fabric_api_base_url=fabric_api_base_url,
                workspace_id=workspace_id,
                headers=headers,
                dataflow_ids=batch_ids,
            )
            print(f"Submitted operation: {operation_id} ({len(batch_ids)} dataflow(s))")

            final_status = PollOperation(fabric_api_base_url, headers, operation_id)
            print(f"Final operation status: {final_status.get('status') if final_status else 'None'}")

            result_payload = GetOperationResult(fabric_api_base_url, headers, operation_id)
            summary_obj = result_payload.get("summary", {}) if isinstance(result_payload, dict) else {}
            result_rows = result_payload.get("migrations", []) if isinstance(result_payload, dict) else []

            if summary_obj:
                print(f"Result summary: {summary_obj}")
            all_result_rows.extend(result_rows)

    vUpgradeSummary = pd.DataFrame(all_result_rows)
    if vUpgradeSummary.empty:
        print("No row-level migration results returned.")
    else:
        print(f"Result rows: {len(vUpgradeSummary)}")
        display(color_code_dataframe_by_status(vUpgradeSummary, status_col="assessmentStatus"))